In [ ]:
# Step 1: Install necessary AI libraries.
# 'transformers' is the core Hugging Face library for LLMs and vision models.
# 'diffusers' is specifically for generative models like Stable Diffusion.
!pip install -q --upgrade transformers==4.56.2 diffusers==0.32.2

In [ ]:
# Step 2: Check for a GPU (Graphics Processing Unit).
# Generative AI requires a lot of math; GPUs are much faster than CPUs for these tasks.
# We use !nvidia-smi to see if a 'Tesla T4' (a common free GPU in Colab) is available.
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)
  if gpu_info.find('Tesla T4') >= 0:
    print("Success - Connected to a T4")
  else:
    print("NOT CONNECTED TO A T4")

In [ ]:
# Step 3: Log into Hugging Face.
# Many models (like SDXL or Flux) require you to accept terms and use an API token.
# This code retrieves your stored token and logs you in automatically.
from huggingface_hub import login
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

In [ ]:
# Step 4: Generate an image using 'SDXL Turbo'.
# Turbo models are optimized to create high-quality images in very few steps (e.g., 4 steps).
# We load the model, move it to the GPU (cuda), and provide a text prompt.
from IPython.display import display
from diffusers import AutoPipelineForText2Image
import torch

pipe = AutoPipelineForText2Image.from_pretrained("stabilityai/sdxl-turbo", torch_dtype=torch.float16, variant="fp16")
pipe.to("cuda")
prompt = "A class of students learning AI engineering in a vibrant pop-art style"
image = pipe(prompt=prompt, num_inference_steps=4, guidance_scale=0.0).images[0]
display(image)

In [ ]:
# Maintenance: Restart the kernel to clear system memory (RAM).
# This is often done to ensure a clean state before loading a larger model.
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
# Step 5: Generate an image using the full SDXL Base model.
# This model is more powerful but slower than the Turbo version (takes ~30 steps).
from IPython.display import display
from diffusers import DiffusionPipeline
import torch

pipe = DiffusionPipeline.from_pretrained("stabilityai/stable-diffusion-xl-base-1.0", torch_dtype=torch.float16, use_safetensors=True, variant="fp16")
pipe.to("cuda")

prompt = "A class of data scientists learning AI engineering in a vibrant high-energy pop-art style"

image = pipe(prompt=prompt, num_inference_steps=30).images[0]

display(image)

In [ ]:
# Maintenance: Shutdown kernel again to free up VRAM for the next model configuration.
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
# Step 6: The Base + Refiner strategy.
# This uses two models: 'Base' creates the structure, and 'Refiner' adds fine details.
# We pass latent data (compressed representation) from the first to the second.
from diffusers import DiffusionPipeline
import torch

base = DiffusionPipeline.from_pretrained("stabilityai/stable-diffusion-xl-base-1.0", torch_dtype=torch.float16, variant="fp16", use_safetensors=True)
base.to("cuda")
refiner = DiffusionPipeline.from_pretrained("stabilityai/stable-diffusion-xl-refiner-1.0", text_encoder_2=base.text_encoder_2, vae=base.vae, torch_dtype=torch.float16, use_safetensors=True, variant="fp16",)
refiner.to("cuda")

n_steps = 40
high_noise_frac = 0.8

prompt = "A class of data scientists learning AI engineering in a vibrant high-energy pop-art style"

image = base(
    prompt=prompt,
    num_inference_steps=n_steps,
    denoising_end=high_noise_frac,
    output_type="latent",
).images

image = refiner(
    prompt=prompt,
    num_inference_steps=n_steps,
    denoising_start=high_noise_frac,
    image=image,
).images[0]

display(image)

In [ ]:
# Maintenance: Clear memory after the dual-model run.
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
# Step 7: Install the 'datasets' library.
# This is used to download training data, voice samples, or text datasets from Hugging Face.
!pip install --upgrade datasets==3.6.0

In [ ]:
# Step 8: Text-to-Speech (TTS) demo.
# This uses a 'pipeline' to convert text into audio using the Microsoft SpeechT5 model.
# It also uses a speaker embedding to define what the voice sounds like.
from transformers import pipeline
from datasets import load_dataset
import soundfile as sf
import torch
from IPython.display import Audio

synthesiser = pipeline("text-to-speech", "microsoft/speecht5_tts", device='cuda')
embeddings_dataset = load_dataset("matthijs/cmu-arctic-xvectors", split="validation", trust_remote_code=True)
speaker_embedding = torch.tensor(embeddings_dataset[7306]["xvector"]).unsqueeze(0)
speech = synthesiser("Hi to an artificial intelligence engineer, on the way to mastery!", forward_params={"speaker_embeddings": speaker_embedding})

Audio(speech["audio"], rate=speech["sampling_rate"])

In [ ]:
# Maintenance: Clear memory to prepare for the high-end Flux model.
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
# Step 9: Verify high-end hardware.
# For advanced models like Flux, a powerful GPU like the NVIDIA A100 is often required.
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)
  if gpu_info.find('A100') >= 0:
    print("Success - Connected to an NVIDIA A100")
  else:
    print("NOT CONNECTED TO AN A100")

In [ ]:
# Step 10: Generate an image using FLUX.1 [schnell].
# Flux is a state-of-the-art model. 'Schnell' is the fast version.
# We track the time taken using the 'datetime' library to estimate costs later.
import torch
from diffusers import FluxPipeline
from IPython.display import display
from datetime import datetime

start = datetime.now()

pipe = FluxPipeline.from_pretrained("black-forest-labs/FLUX.1-schnell", torch_dtype=torch.bfloat16).to("cuda")
generator = torch.Generator(device="cuda").manual_seed(0)
prompt = "A class of data scientists learning AI engineering in a vibrant high-energy pop-art style"

image = pipe(
    prompt,
    guidance_scale=0.0,
    num_inference_steps=4,
    max_sequence_length=256,
    generator=generator
).images[0]

display(image)

stop = datetime.now()

In [ ]:
# Step 11: Calculate the cost of the run.
# Cloud GPUs (like A100s) have an hourly rate.
# This cell converts the processing time into a dollar amount based on current rates.
seconds = (stop-start).total_seconds()
units_per_hour = 5.37
estimated_units = (5.37 / 3600) * seconds
estimated_cost = estimated_units * (9.99/100)
print(f"This took {seconds:.1f} seconds and cost an estimated ${estimated_cost:.3f}")